In [50]:
from datetime import date
import hisepy
import mudata as md
import os
import pandas as pd
import polars
import snapatac2 as snap
import subprocess

In [2]:
if not os.path.isdir('output'):
    os.mkdir('output')

## Helper functions

In [3]:
def read_feat(in_file):
    with open(in_file, "r") as file:
        feat = [line.rstrip() for line in file]
        header = feat.pop(0)
    return feat

In [4]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

## Arrow read functions

In [5]:
import h5py
import scipy.sparse as scs
from collections import Counter
import multiprocessing
import itertools
import re
import numpy

In [6]:
def missing_seq(seq, max_val):
    value_set = set(range(0, max_val))
    missing = value_set.difference(seq)
    return sorted(missing)

def read_matrix(arrow_file, target, binary = False):
    arrow = h5py.File(arrow_file, 'r')
    
    feats = arrow[target]['Info']['FeatureDF'][:]
    feat_chroms = [feat[0] for feat in feats]
    feat_chroms = [x.decode('UTF-8') for x in feat_chroms]
    chrom_counts = Counter(feat_chroms)
    
    obs = arrow[target]['Info']['CellNames'][:]
    
    chroms = []
    for k in arrow[target].keys():
        if k != "Info":
            chroms.append(k)
    
    chrom_mats = []
    for chrom in chroms:
        
        if binary:
            data = [int(1)] * arrow[target][chrom]['i'].shape[1]
        else:
            data = arrow[target][chrom]['x'][:][0]

        indices = arrow[target][chrom]['i'][:][0]
        indices = indices - 1
        
        included_indices = arrow[target][chrom]['jValues'][:] - 1
        missing_indices = missing_seq(included_indices[0], obs.shape[0])
        
        indlens = arrow[target][chrom]['jLengths'][:][0]
        indlens = numpy.insert(indlens, missing_indices, 0)

        indptr = numpy.cumsum(indlens)
        indptr = numpy.insert(indptr, 0, 0)

        chrom_mat = scs.csc_matrix((data, indices, indptr),
                                   shape = (chrom_counts[chrom], obs.shape[0]))

        chrom_mats.append(chrom_mat)

    mat = scs.vstack(chrom_mats, format = 'csc')
    
    arrow.close()
    
    return mat

In [7]:
def read_feats(arrow_file, target):
    arrow = h5py.File(arrow_file, 'r')
    feats = arrow[target]['Info']['FeatureDF'][:]
    feats = polars.DataFrame(feats)
    feats = feats.with_columns(
        polars.Series(
            name = 'seqnames',
            values = [x.decode('UTF-8') for x in feats['seqnames']]
        )
    )

    if not 'name' in feats.columns:
        feats = feats.with_columns(
            polars.concat_str(
                [polars.col('seqnames'), polars.col('start')],
                separator = '_'
            ).alias('name')
        )
    else:
        feats = feats.with_columns(
            polars.Series(
                name = 'name',
                values = [x.decode('UTF-8') for x in feats['name']]
            )
        )
    
    arrow.close()
    
    return(feats)

In [8]:
def read_obs(arrow_file):
    arrow = h5py.File(arrow_file, 'r')

    obs_keys = arrow['Metadata'].keys()

    obs_dict = {}
    for key in obs_keys:
        values = arrow['Metadata'][key][:].tolist()

        if len(values) == 1:
            continue

        if(isinstance(values[0], (bytes, bytearray))):
            # Decode byte strings
            values = [x.decode('UTF-8') for x in values]

        obs_dict[key] = values

    arrow.close()

    obs_df = polars.DataFrame(obs_dict)

    return obs_df

In [9]:
def read_bc_idx(h5, chrom):
    bc = h5['Metadata']['barcodes']
    bc = [x.decode('utf8') for x in bc]
    cbc = h5['Metadata']['CellNames'][:].tolist()

    cbc_bc_dict = {}
    for i in range(len(bc)):
        cbc_bc_dict[cbc[i]] = bc[i]
    
    fr_cbc = h5['Fragments'][chrom]['RGValues'][:]
    if len(fr_cbc) == 1:
        fr_cbc = fr_cbc[0]
    
    fr_bc = [cbc_bc_dict[x] for x in fr_cbc]
    
    fr_lens = h5['Fragments'][chrom]['RGLengths']
    
    cs = numpy.cumsum(fr_lens)
    cs = numpy.insert(cs,[0],0)
    st = cs[0:(len(cs) - 1)]
    en = cs[1:]

    n = []
    for i in range(len(st)):
        n.append(en[i] - st[i])
    
    res = polars.DataFrame({'bc': fr_bc,
                            'st': st,
                            'en': en,
                            'n': n})
    
    return res

def read_all_fragments_chr(h5_file, chrom, sort):
    h5 = h5py.File(h5_file, 'r')
    
    bc_idx = read_bc_idx(h5, chrom)
    
    h5_ranges = h5['Fragments'][chrom]['Ranges']
    
    h5_st = h5_ranges[0][:]
    h5_wi = h5_ranges[1][:]
    h5_en = [h5_st[i] + h5_wi[i] for i in range(len(h5_st))]
    
    res_bc = []
    
    for x in range(bc_idx.shape[0]):
        res_bc.extend([bc_idx['bc'][x]] * bc_idx['n'][x])
    
    res = polars.DataFrame({'chr': [chrom] * len(h5_st),
                            'start': h5_st,
                            'end': h5_en,
                            'barcode': res_bc,
                            'score': [1] * len(h5_st)})
    
    if sort:
        res = res.sort('start')
    
    return res

def read_all_fragments(h5_file, n_threads):
    pool = multiprocessing.Pool(n_threads)
    
    h5 = h5py.File(h5_file, 'r')
    chroms = h5['Fragments'].keys()
    chroms = [x for x in chroms]
        
    param_list = zip(itertools.repeat(h5_file),
                     chroms,
                     itertools.repeat(False))
    
    res = pool.starmap(read_all_fragments_chr, param_list)
    
    pool.close()
    
    out_res = polars.concat(res)
    
    return out_res

def write_fragments_tsv(frag, filename):
    frag.write_csv(
        filename,
        separator = '\t',
        include_header = False)

## Get Arrow files from HISE

In [10]:
arrow_uuids = {
    'untreated-t0':      'ad2f347d-e961-4a70-b294-4df83c12355d',
    'dmso-t4':           '8c2a93be-de53-4d6f-ae2e-ab40a356edc3',
    'dmso-t24':          '9ab975f8-7763-4892-96a7-a7438ecc9470',
    'dmso-t72':          '16e0c562-5d36-431f-bb27-b443aabc7077',
    'bortezomib-t4':     'c299a55a-3325-4eb3-ba8e-c8ceccafaa8c',
    'bortezomib-t24':    'a56fd2ba-a055-4ff8-9ab1-69df113bc032',
    'bortezomib-t72':    '57dde81e-bdaa-4138-add6-2551968672f4',
    'lenalidomide-t4':   '6d8185bf-8a35-492d-a6ab-5783006b3b8e',
    'lenalidomide-t24':  'ff8fe67e-cfe0-482a-bad1-aa189390a1c0',
    'lenalidomide-t72':  '052b769d-cbdf-41f6-8fe8-0d34564b442a',
    'dexamethasone-t4':  '6853fb68-fa85-43d5-9071-c5e42667a75e',
    'dexamethasone-t24': '30167a93-70a8-4c38-b615-2252dabe417e'
}

In [11]:
arrow_files = {}
for k, uuid in arrow_uuids.items():
    arrow_files[k] = hisepy.cache_files([uuid])[0]

2026-06-11 15:43:10,168 INFO [hisepy.logging:175] logging 1410 136648083720000 Calling cache_files
2026-06-11 15:43:18,827 INFO [hisepy.logging:208] logging 1410 136648083720000 Finished cache_files, success=True, time_elapsed=5.061s
2026-06-11 15:43:18,828 INFO [hisepy.logging:175] logging 1410 136648083720000 Calling cache_files
2026-06-11 15:43:26,065 INFO [hisepy.logging:208] logging 1410 136648083720000 Finished cache_files, success=True, time_elapsed=4.100s
2026-06-11 15:43:26,066 INFO [hisepy.logging:175] logging 1410 136648083720000 Calling cache_files
2026-06-11 15:43:32,446 INFO [hisepy.logging:208] logging 1410 136648083720000 Finished cache_files, success=True, time_elapsed=4.393s
2026-06-11 15:43:32,447 INFO [hisepy.logging:175] logging 1410 136648083720000 Calling cache_files
2026-06-11 15:43:38,434 INFO [hisepy.logging:208] logging 1410 136648083720000 Finished cache_files, success=True, time_elapsed=3.801s
2026-06-11 15:43:38,435 INFO [hisepy.logging:175] logging 1410 1

## Get h5mu from HISE

In [12]:
h5mu_uuid = '59fee21d-3170-42a6-b70d-84de24e7f73c'
h5mu = hisepy.cache_files([h5mu_uuid])[0]
mdata = md.read_h5mu(h5mu)

2026-06-11 15:44:24,333 INFO [hisepy.logging:175] logging 1410 136648083720000 Calling cache_files
2026-06-11 15:44:28,561 INFO [hisepy.logging:208] logging 1410 136648083720000 Finished cache_files, success=True, time_elapsed=2.660s


/home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/python3.13/site-packages/mudata/_core/mudata.py:1416: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("var", axis=0, join_common=join_common)
/home/workspace/environment/archrpixiv11/.pixi/envs/default/lib/python3.13/site-packages/mudata/_core/mudata.py:1272: FutureWarning: From 0.4 .update() will not pull obs/var columns from individual modalities by default anymore. Set mudata.set_options(pull_on_update=False) to adopt the new behaviour, which will become the default. Use new pull_obs/pull_var and push_obs/push_var methods for more flexibility.
  self._update_attr("obs", axis=1, join_common=join_common)


## Extract fragments

We'll also filter against the .h5mu, which is only the labeled cells.

In [13]:
frag_files = {}
for k, arrow in arrow_files.items():
    out_file = f'output/tcell-vrd_{k}_labeled_data_fragments.tsv'
    if not os.path.isfile(out_file):
        frags = read_all_fragments(arrow, n_threads = 12)
        frags = frags.filter(polars.col('barcode').is_in(mdata.obs_names.to_list()))
        frags = frags.sort('barcode')
        write_fragments_tsv(frags, out_file)
    frag_files[k] = out_file

## Build SnapATAC2 AnnData objects

These can be integrated into our .h5mu to include the full set of TEA-seq multimodal data.

In [21]:
snaps = {}
for k, arrow in arrow_files.items():
    mat = read_matrix(arrow, target = 'TileMatrix', binary = True)
    mat = mat.astype('uint16')
    obs = read_obs(arrow)
    mat_bc = obs['barcodes'].to_list()

    snap_data = snap.pp.import_fragments(
        fragment_file = frag_files[k],
        chrom_sizes = snap.genome.hg38,
        sorted_by_barcode = True
    )

    keep_lgl = [x in snap_data.obs_names for x in mat_bc]
    mat = mat[:,keep_lgl]
    mat_bc = list(itertools.compress(mat_bc, keep_lgl))

    snap_data = snap_data[mat_bc].copy()
    
    snap.pp.add_tile_matrix(snap_data)
    snap_data.X = mat.transpose()
    
    snaps[k] = snap_data

IOStream.flush timed out


In [22]:
snap_files = {}
for k, snap_data in snaps.items():
    out_file = f'output/tcell-vrd_{k}_labeled_data_snap.h5ad'
    if not os.path.isfile(out_file):
        snap_data.write(out_file)
    snap_files[k] = out_file

In [23]:
snap_mats = {}
snap_obs = {}
for k, snap_data in snaps.items():
    snap_mats[k] = snap_data.X
    snap_obs[k] = snap_data.obs

In [24]:
del(snaps)

## Combine data to get the full set for .h5mu incorporation

Simply using snap.concat doesn't seem to work. We'll combine all of the fragments and build a new object.

In [17]:
sub_call = ['cat'] + list(frag_files.values()) + ['>', 'output/tcell-vrd_all_labeled_data_fragments.tsv']
sub_call = ' '.join(sub_call)

In [18]:
subprocess.run(sub_call, shell = True)

CompletedProcess(args='cat output/tcell-vrd_untreated-t0_labeled_data_fragments.tsv output/tcell-vrd_dmso-t4_labeled_data_fragments.tsv output/tcell-vrd_dmso-t24_labeled_data_fragments.tsv output/tcell-vrd_dmso-t72_labeled_data_fragments.tsv output/tcell-vrd_bortezomib-t4_labeled_data_fragments.tsv output/tcell-vrd_bortezomib-t24_labeled_data_fragments.tsv output/tcell-vrd_bortezomib-t72_labeled_data_fragments.tsv output/tcell-vrd_lenalidomide-t4_labeled_data_fragments.tsv output/tcell-vrd_lenalidomide-t24_labeled_data_fragments.tsv output/tcell-vrd_lenalidomide-t72_labeled_data_fragments.tsv output/tcell-vrd_dexamethasone-t4_labeled_data_fragments.tsv output/tcell-vrd_dexamethasone-t24_labeled_data_fragments.tsv > output/tcell-vrd_all_labeled_data_fragments.tsv', returncode=0)

In [19]:
full_snap = snap.pp.import_fragments(
    fragment_file = 'output/tcell-vrd_all_labeled_data_fragments.tsv',
    chrom_sizes = snap.genome.hg38,
    sorted_by_barcode = True
)

In [20]:
full_snap

AnnData object with n_obs × n_vars = 147414 × 0
    obs: 'n_fragment', 'frac_dup', 'frac_mito'
    uns: 'reference_sequences'
    obsm: 'fragment_paired'

In [29]:
full_snap_mat = scs.vstack(list(snap_mats.values()))

In [30]:
full_snap_mat

<Compressed Sparse Row sparse matrix of dtype 'uint16'
	with 1501047058 stored elements and shape (147414, 6062095)>

In [34]:
full_snap_obs = pd.concat(list(snap_obs.values()))

In [36]:
full_snap_obs.head()

,n_fragment,frac_dup,frac_mito
42191abafb7b11ed8a1abe9d4c5fcb23,19989,0.0,0.0
439b1870fb7b11ed933dc6f18f77c1a7,19938,0.0,0.0
9cd001acfb8111ed99031af2246d8fa2,19837,0.0,0.0
cc541db8fb8711edbbcffe1e90e63656,19796,0.0,0.0
b4101404fb7e11ed84ffaefa3cccb57a,19755,0.0,0.0


In [38]:
full_snap = full_snap[full_snap_obs.index.to_list()].copy()

In [41]:
snap.pp.add_tile_matrix(full_snap)

In [42]:
full_snap.X = full_snap_mat

In [43]:
full_snap_file = f'output/tcell-vrd_all_labeled_data_snap.h5ad'
full_snap.write(full_snap_file)

## Integrate into the mdata object

In [45]:
full_snap = full_snap[mdata.obs_names.to_list()].copy()

In [46]:
mdata.mod['atac'] = full_snap

In [ ]:
out_h5mu = 'output/tcell-vrd_all_labeled_data_tea_{d}.h5mu'.format(d = date.today())
mdata.write(out_h5mu)